In [ ]:
"""NEXUS local AI bridge — standalone Python 3.10+, standard library only.
Dean A. Kulik / QuHarmonics Research Group: native partner-ring operation.
Prototype integration and tests; not a claim about physical realization.

Usage:
  1. In LM Studio: load one model; Developer > Start Server.
  2. python NEXUS_local_AI_bridge.py
  3. Try ':state', ':step', ':reverse', or ask the AI to do those operations.

No shell execution, filesystem browsing, or network calls beyond local LM Studio.
An append-only local event log is replayed and verified at next startup.
"""
from __future__ import annotations
import json
import os
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

BASE = os.getenv('LM_STUDIO_URL', 'http://localhost:1235/v1').rstrip('/')
# LM_STUDIO_URL may point to either the server root or its /v1 API prefix.
if BASE.endswith('/v1'):
    BASE = BASE[:-3]
TOKEN = os.getenv('LM_API_TOKEN', '')
WIDTH = 4
SEED_P = 0
SEED_W = 1
# Jupyter cells do not define __file__; use the notebook working directory there.
# Standalone scripts keep the ledger beside the script.
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
LEDGER = ROOT / 'NEXUS_local_AI_events.jsonl'


def partner(x: int, n: int) -> int:
    """Native P: bit reversal except palindromes, which complement."""
    mask = (1 << n) - 1
    if not (2 <= n <= 16 and 0 <= x <= mask):
        raise ValueError('n must be 2..16 and x must fit n bits')
    rev = int(f'{x:0{n}b}'[::-1], 2)
    return rev if rev != x else x ^ mask


class Machine:
    def __init__(self, n=WIDTH, p=SEED_P, w=SEED_W):
        self.n = n
        self.p = p
        self.w = w
        self.D: dict[int, int] = {}

    def snapshot(self):
        return {'n': self.n, 'p': self.p, 'w': self.w,
                'field': {str(k): v for k, v in sorted(self.D.items())}}

    def _write(self, at: int, value: int):
        if value:
            self.D[at] = value
        else:
            self.D.pop(at, None)

    def forward(self):
        # x = D[p]^w; p' = p^x; w' = P(x); D'[p] = D[p]^w'
        oldp = self.p
        x = self.D.get(oldp, 0) ^ self.w
        w2 = partner(x, self.n)
        self._write(oldp, self.D.get(oldp, 0) ^ w2)
        self.p = oldp ^ x
        self.w = w2
        return {'x': x, 'written_at': oldp, 'direction': 'forward'}

    def backward(self):
        # x=P(w'); p=p'^x; D[p]=D'[p]^w'; w=x^D[p]
        x = partner(self.w, self.n)
        oldp = self.p ^ x
        oldw = self.w
        recovered = self.D.get(oldp, 0) ^ oldw
        self._write(oldp, recovered)
        self.p = oldp
        self.w = x ^ recovered
        return {'x': x, 'restored_at': oldp, 'direction': 'reverse'}


def verify():
    # Exact P involution, and exact forward/backward return at fixed seeds.
    for n in range(2, 11):
        for x in range(1 << n):
            assert partner(partner(x, n), n) == x
            assert partner(x, n) != x
    for n in (2, 4, 8, 16):
        m = Machine(n=n, p=0, w=1)
        initial = m.snapshot()
        for _ in range(24):
            before = m.snapshot()
            m.forward()
            after = m.snapshot()
            m.backward()
            assert m.snapshot() == before, (n, 'roundtrip')
            m.forward()
            assert m.snapshot() == after, (n, 'inverse roundtrip')
        for _ in range(24):
            m.backward()
        assert m.snapshot() == initial, (n, '24-step return')
    print('RUNS: P involution at widths 2..10; 24-step exact returns at widths 2,4,8,16.')


def replay():
    m = Machine()
    if not LEDGER.exists():
        return m
    count = 0
    with LEDGER.open('r', encoding='utf-8') as f:
        for count, line in enumerate(f, 1):
            event = json.loads(line)
            if event['before'] != m.snapshot():
                raise RuntimeError(f'Ledger discontinuity before line {count}; no state changed')
            if event['operation'] == 'step':
                m.forward()
            elif event['operation'] == 'reverse':
                m.backward()
            else:
                raise RuntimeError(f'Unexpected ledger operation on line {count}')
            if event['after'] != m.snapshot():
                raise RuntimeError(f'Ledger mismatch after line {count}; no state changed')
    print(f'Replayed and verified {count} prior event(s).')
    return m


def operation(m: Machine, kind: str):
    if kind == 'state':
        return {'status': 'RUNS', 'state': m.snapshot()}
    if kind not in ('step', 'reverse'):
        return {'error': 'Only state, step, and reverse are allowed'}
    before = m.snapshot()
    meta = m.forward() if kind == 'step' else m.backward()
    after = m.snapshot()
    # A local independent way-home assertion, not a model-generated claim.
    check = Machine(n=m.n, p=m.p, w=m.w)
    check.D = dict(m.D)
    (check.backward() if kind == 'step' else check.forward())
    assert check.snapshot() == before, 'Return path failed'
    event = {'operation': kind, 'before': before, 'after': after, 'route': meta}
    with LEDGER.open('a', encoding='utf-8') as f:
        f.write(json.dumps(event, sort_keys=True) + '\n')
    return {'status': 'RUNS', **event, 'return_verified': True}


def request_json(method: str, endpoint: str, payload=None, timeout=180):
    headers = {'Content-Type': 'application/json'}
    if TOKEN:
        headers['Authorization'] = f'Bearer {TOKEN}'
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = Request(BASE + endpoint, headers=headers, data=data, method=method)
    with urlopen(req, timeout=timeout) as res:
        return json.loads(res.read().decode('utf-8'))


def find_model():
    # Unlike /v1/models, /api/v1/models explicitly identifies loaded instances.
    data = request_json('GET', '/api/v1/models')
    loaded = [(model['display_name'], inst['id'])
              for model in data.get('models', []) if model.get('type') == 'llm'
              for inst in model.get('loaded_instances', [])]
    if not loaded:
        print('No loaded LLM. Load Gemma 4 E4B in LM Studio, then rerun.')
        return None
    for i, (label, ident) in enumerate(loaded, 1):
        print(f'  {i}. {label}  [{ident}]')
    print('Using first loaded model:', loaded[0][0])
    return loaded[0][1]


TOOLS = [{'type': 'function', 'function': {
    'name': name,
    'description': description,
    'parameters': {'type': 'object', 'properties': {}, 'additionalProperties': False}
}} for name, description in (
    ('nexus_state', 'Read exact current binary reader, carry, and sparse field.'),
    ('nexus_step', 'Advance one exact reversible NEXUS reader-field step.'),
    ('nexus_reverse', 'Apply exact inverse of one NEXUS reader-field step.'),
)]


def ask_model(model_id: str, machine: Machine, user_text: str):
    messages = [
        {'role': 'system', 'content': (
            'You are an interface to a local deterministic NEXUS ring machine. '
            'For actual machine state or changes, use the offered tools. '
            'Do not invent tool outputs. The engine owns state and verification. '
            'When reporting results, distinguish exact computation from interpretation.')},
        {'role': 'user', 'content': user_text},
    ]
    for _ in range(5):
        response = request_json('POST', '/v1/chat/completions', {
            'model': model_id, 'messages': messages, 'tools': TOOLS,
            'tool_choice': 'auto', 'temperature': 0.1, 'max_tokens': 512,
            'stream': False,
        })
        msg = response['choices'][0]['message']
        calls = msg.get('tool_calls') or []
        if not calls:
            print('\nMODEL:', msg.get('content') or '(no text returned)')
            return
        messages.append({'role': 'assistant', 'content': msg.get('content'),
                         'tool_calls': calls})
        for call in calls[:4]:
            fn = call.get('function', {}).get('name', '')
            kind = {'nexus_state':'state', 'nexus_step':'step',
                    'nexus_reverse':'reverse'}.get(fn, '')
            result = operation(machine, kind)
            print(f'  TOOL {fn}:', json.dumps(result, sort_keys=True))
            messages.append({'role': 'tool', 'tool_call_id': call['id'],
                             'content': json.dumps(result)})
    print('Tool-call round limit reached; state:', machine.snapshot())


def main():
    verify()
    machine = replay()
    print('Current state:', json.dumps(machine.snapshot(), sort_keys=True))
    print('LM Studio API root:', BASE)
    try:
        model_id = find_model()
    except (URLError, HTTPError, TimeoutError) as ex:
        print('LM Studio API not reachable:', ex)
        print('Start LM Studio > Developer > Start Server; load one model.')
        model_id = None
    print("Commands: :state, :step, :reverse, :quit. Or type a model request.")
    while True:
        try:
            line = input('nexus> ').strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not line: continue
        if line == ':quit': break
        if line in (':state', ':step', ':reverse'):
            print(json.dumps(operation(machine, line[1:]), indent=2))
        elif not model_id:
            print('Model unavailable. Use :step/:reverse or start the LM Studio server.')
        else:
            try:
                ask_model(model_id, machine, line)
            except (URLError, HTTPError, TimeoutError, KeyError) as ex:
                print('Model API error:', ex)


if __name__ == '__main__':
    main()


RUNS: P involution at widths 2..10; 24-step exact returns at widths 2,4,8,16.
Current state: {"field": {}, "n": 4, "p": 0, "w": 1}
LM Studio API root: http://localhost:1235
  1. Gemma 4 E4B  [google/gemma-4-e4b]
Using first loaded model: Gemma 4 E4B
Commands: :state, :step, :reverse, :quit. Or type a model request.


nexus>  :state


{
  "status": "RUNS",
  "state": {
    "n": 4,
    "p": 0,
    "w": 1,
    "field": {}
  }
}


nexus>  :step


{
  "status": "RUNS",
  "operation": "step",
  "before": {
    "n": 4,
    "p": 0,
    "w": 1,
    "field": {}
  },
  "after": {
    "n": 4,
    "p": 1,
    "w": 8,
    "field": {
      "0": 8
    }
  },
  "route": {
    "x": 1,
    "written_at": 0,
    "direction": "forward"
  },
  "return_verified": true
}


nexus>  reverse


  TOOL nexus_reverse: {"after": {"field": {}, "n": 4, "p": 0, "w": 1}, "before": {"field": {"0": 8}, "n": 4, "p": 1, "w": 8}, "operation": "reverse", "return_verified": true, "route": {"direction": "reverse", "restored_at": 0, "x": 1}, "status": "RUNS"}

MODEL: The exact computation for the reverse operation has been performed:

**Operation:** `nexus_reverse`
**Status:** Success (`RUNS`)

**State Before:**
*   $n$: 4
*   $p$: 1
*   $w$: 8
*   Field: `{ "0": 8 }`

**State After:**
*   $n$: 4
*   $p$: 0
*   $w$: 1
*   Field: `{}`

The operation was verified as successful.


nexus>  Read the current Nexus state, advance one step, reverse it, and verify that we return to the original state.


  TOOL nexus_state: {"state": {"field": {}, "n": 4, "p": 0, "w": 1}, "status": "RUNS"}
  TOOL nexus_step: {"after": {"field": {"0": 8}, "n": 4, "p": 1, "w": 8}, "before": {"field": {}, "n": 4, "p": 0, "w": 1}, "operation": "step", "return_verified": true, "route": {"direction": "forward", "written_at": 0, "x": 1}, "status": "RUNS"}
  TOOL nexus_reverse: {"after": {"field": {}, "n": 4, "p": 0, "w": 1}, "before": {"field": {"0": 8}, "n": 4, "p": 1, "w": 8}, "operation": "reverse", "return_verified": true, "route": {"direction": "reverse", "restored_at": 0, "x": 1}, "status": "RUNS"}

MODEL: The requested sequence of operations has been completed:

1.  **Read Initial State:** The initial Nexus state was read successfully.
    *   State: `{"n": 4, "p": 0, "w": 1, "field": {}}`

2.  **Advance One Step (`nexus_step`):** The machine advanced one step.
    *   The state changed from the initial reading to a new configuration (details provided in the tool output).

3.  **Reverse the Step (`nexu